---

# University of Liverpool

## COMP534 - Applied AI

---

This notebook is associated with Assignment 1. Use it to complete the assignment by following the instructions provided in each section, which includes a text cell describing the requirements. For additional details, see the Canvas.

Use this first cell to import the necessary libraries.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC

# 1. **Data Management**


In this part, you need to:

1.   analyse and prepare the data. Use plots, graphs, and tables (such as histogram, box plots, scatterplots, etc.) to better analyse the dataset and identify issues or potential improvements in the data, including (but not limited to) unnecessary feature/variable which can be dropped/removed, standardization, encoding, etc;
2.	define an appropriate experimental protocol (such as cross-validation, k-fold, etc).


In [ ]:
# 1.1 Overview of the dataset

# Load the target file into DataFrames and get an overview of the dataset
df = pd.read_csv('dataset_prepared.csv')
df.info()
df.describe()

# Check missing values and duplications.
df.isnull().sum()
df.duplicated(subset=['id']).sum()

# Explore the distribution of the target variable.
diagnosis_colors = {'CI': '#6FB07F',
                    'Depression': '#FCB03C', 'Both': '#FC5B3F'}
sns.countplot(data=df, x='diagnosis', hue='diagnosis', legend=False,
              palette=diagnosis_colors, width=0.5)
# plt.title("Distribution of Diagnosis")
sns.despine()
plt.show()

In [ ]:
# 1.2 Remove unnecessary features (“id” and “country”).

df_cleaned_1= df.drop(columns=['id', 'country'])
df_cleaned_1.info()

In [ ]:
# 1.3 Perform EDA on numerical features, comparing the distributions across three classes and detecting potential outliers.

def plot_eda(df, features, target='diagnosis'):
    """Plots histograms and boxplots for each numerical feature, comparing the distributions across classes."""
    n_vars = len(features)
    categories = ["CI", "Depression", "Both"]

    fig = plt.figure(figsize=(7, n_vars * 2.5))
    outer_gs = gridspec.GridSpec(n_vars, 1, figure=fig, hspace=0.5)

    for i, col in enumerate(features):
        inner_gs = gridspec.GridSpecFromSubplotSpec(3, 2, subplot_spec=outer_gs[i],
                                                    wspace=0.25, hspace=0.3)
        x_min, x_max = df[col].min(), df[col].max()
        pad = (x_max - x_min) * 0.05
        xlim = (x_min - pad, x_max + pad)

        # --- left: histogram ---
        for j, cat in enumerate(categories):
            ax_hist = fig.add_subplot(inner_gs[j, 0])
            subset = df[df[target] == cat]

            sns.histplot(data=subset, x=col, kde=True, color=diagnosis_colors[cat],
                         ax=ax_hist, alpha=0.6, element="bars")
            ax_hist.set_xlim(xlim)
            ax_hist.set_ylabel(
                cat, fontweight='bold', color=diagnosis_colors[cat], rotation=0, labelpad=40, va='center')
            
            sns.despine(ax=ax_hist)

        # --- right：boxplot ---
        ax_box = fig.add_subplot(inner_gs[:, 1])
        sns.boxplot(data=df, x=col, y=target, order=categories,
                    ax=ax_box, hue=target, palette=diagnosis_colors, legend=False, width=0.5, dodge=False)
        ax_box.set_xlim(xlim)
        ax_box.set_ylabel("") 
        ax_box.set_yticklabels([])
        sns.despine(ax=ax_box)

    plt.suptitle("Multi-Class Feature Distribution Analysis\n(Left: Histograms, Right: Boxplots)",
                 fontsize=12, fontweight='bold', y=0.91)
    plt.show()


num_features = ['age', 'sleep_quality_index', 'deep_sleep_quality_index', 'brain_fog_level', 'physical_pain_score', 'stress_level',
                'depression_phq9_score', 'fatigue_severity_scale_score', 'pem_duration_hours', 'hours_of_sleep_per_night']
plot_eda(df_cleaned_1, num_features)

# Based on the EDA, we can identify potential outliers in the 'depression_phq9_score' feature.
# The boxplot for this feature shows that there are some values that exceed the typical range of scores for the PHQ-9 assessment (0-27).
high_outliers_phq9 = df_cleaned_1[df_cleaned_1['depression_phq9_score'] > 27]
print("Potential outliers in 'depression_phq9_score':")
print(high_outliers_phq9[['depression_phq9_score', 'diagnosis']])

# Remove outliers.
df_cleaned_2 = df_cleaned_1[df_cleaned_1['depression_phq9_score'] <= 27]
df_cleaned_2.info()

In [ ]:
# 1.4 Correlation analysis of numerical features and remove highly correlated features.

# Compute the correlation matrix and visualize it using a heatmap.
corr_matrix = df_cleaned_2[num_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".1f", cmap='coolwarm')
plt.show()

# Remove the 'deep_sleep_quality_index' column due to its high correlation with 'total_sleep_time_minutes'.
df_cleaned_3 = df_cleaned_2.drop(columns='deep_sleep_quality_index')
df_cleaned_3.info()

In [ ]:
# 1.5 Encoding.

# Encode the target variable 'diagnosis' using label encoding.
diagnosis_mapping = {'CI': 0, 'Depression': 1, 'Both': 2}
df_cleaned_3['diagnosis'] = df_cleaned_3['diagnosis'].replace(
    diagnosis_mapping).astype(int)

# Encode ordinal variables using label encoding.
work_status_mapping = {'Not working': 0, 'Partially working': 1, 'Working': 2}
df_cleaned_3['work_status'] = df_cleaned_3['work_status'].replace(work_status_mapping)

social_activity_level_mapping = {'Very low': 0, 'Low': 1, 'Medium': 2,'High': 3,'Very high': 4}
df_cleaned_3['social_activity_level'] = df_cleaned_3['social_activity_level'].replace(social_activity_level_mapping)

exercise_frequency_mapping = {'Never': 0, 'Rarely': 1, 'Sometimes': 2,  'Often': 3, 'Daily': 4}
df_cleaned_3['exercise_frequency'] = df_cleaned_3['exercise_frequency'].replace(exercise_frequency_mapping)

# Encode nominal variables using one-hot encoding.
df_cleaned_4 = pd.get_dummies(df_cleaned_3, columns=['gender', 'meditation_or_mindfulness'], dtype=int)

print(df_cleaned_4)
df_cleaned_4.info()

In [ ]:
# 1.6 Data split.

# Prepare the feature matrix X and target vector y, then split the data into training and testing sets with stratification to maintain class distribution.
X = df_cleaned_4.drop(columns='diagnosis')
y = df_cleaned_4['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,
                                                    random_state=42, stratify=y)

---

# 2. **Model Training**

Here, you need to:

1.	select and compare at least three machine learning models (seen/discussed during the lectures) appropriate for your modelling;
2.	if there are hyperparameters in a selected algorithm, define a hyperparameter search protocol (you can define your own), and tune them.


In [ ]:
# 2.1 Random Forest

# 1. Instantiating the Random Forest
rf = RandomForestClassifier(random_state=42)

# 2. Define the hyperparameter grid
para_grid_rf = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 3, 10],
    'min_samples_leaf': [2, 10, 50]}

# 3. Set up GridSearchCV
# 5-fold cross-validation is used to evaluate the performance of each hyperparameter combination
# The best model is selected based on the highest macro recall score. 
scoring = ['recall_macro', 'precision_macro', 'f1_macro']
grid_rf = GridSearchCV(estimator=rf, param_grid=para_grid_rf,
                       scoring=scoring, refit='recall_macro', cv=5, n_jobs=-1, verbose=2)

# 4. Fit the grid search to the training data
grid_rf.fit(X_train, y_train)

# 5. Extract top 5 and last 5 hyperparameter combinations
cv_results_rf = pd.DataFrame(grid_rf.cv_results_).sort_values(
    by=['rank_test_recall_macro'])

cv_results_rf_top5 = cv_results_rf.head(5)
cv_results_rf_last5 = cv_results_rf.tail(5)

cv_results_rf_top5 = cv_results_rf_top5.set_index(
    'params').rename_axis('Hyperparameters')
cv_results_rf_last5 = cv_results_rf_last5.set_index(
    'params').rename_axis('Hyperparameters')

cols_to_show = ['mean_test_recall_macro', 'mean_test_precision_macro', 'mean_test_f1_macro']
cv_results_rf_top5 = cv_results_rf_top5[cols_to_show]
cv_results_rf_last5 = cv_results_rf_last5[cols_to_show]

print("--- Top 5 Hyperparameter Combinations ---")
print(cv_results_rf_top5)
print("\n--- Last 5 Hyperparameter Combinations ---")
print(cv_results_rf_last5)
print("\nBest Hyperparameters:", grid_rf.best_params_)
print(
    "Best Cross-Validation Score (recall_macro): {:.2f}".format(grid_rf.best_score_))

In [ ]:
# 2.2 Logistic Regression

# 1. Instantiate the LogisticRegression model.
lr = LogisticRegression(random_state=42)

# 2. Standardize features.
num_features_cleaned = ['age', 'sleep_quality_index', 'brain_fog_level', 'physical_pain_score', 'stress_level', 'depression_phq9_score', 'fatigue_severity_scale_score', 'pem_duration_hours', 'hours_of_sleep_per_night']
# num_features_cleaned = num_features.remove('deep_sleep_quality_index')
cat_features_cleaned = ['pem_present', 'work_status',
                'social_activity_level', 'exercise_frequency', 'meditation_or_mindfulness']

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_features_cleaned)], remainder='passthrough')

pipe = make_pipeline(preprocessor, lr)

# 3. Define the hyperparameter grid.
param_grid_lr = {
    'logisticregression__C': [0.001, 0.01, 0.1, 1],
    'logisticregression__penalty': ['l1', 'l2'],
    'logisticregression__solver': ['saga']
}
# 4. Set up GridSearchCV.
grid_lr = GridSearchCV(
    estimator=pipe, param_grid=param_grid_lr, scoring=scoring, refit='recall_macro', cv=5, n_jobs=-1, verbose=2)

# 5. Fit the grid search to the training data.
grid_lr.fit(X_train, y_train)

# 6. Extract performances of different hyperparameter combinations.
cv_results_lr = pd.DataFrame(grid_lr.cv_results_).sort_values(
    by=['rank_test_recall_macro'])

cv_results_lr = cv_results_lr.set_index(
    'params').rename_axis('Hyperparameters')

cv_results_lr = cv_results_lr[cols_to_show]

print("--- Performances of different Hyperparameter Combinations ---")
print(cv_results_lr)

print("\nBest Hyperparameters:", grid_lr.best_params_)
print(
    "Best Cross-Validation Score (recall_macro): {:.2f}".format(grid_lr.best_score_))

In [ ]:
# 2.3 SVM

# 1. Instantiate the SVM model.
sv = SVC(random_state=42)

# 2. Standardize features.
pipe = make_pipeline(preprocessor, sv)

# 3. Define the hyperparameter grid.
param_grid_sv = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['rbf'],
    'svc__gamma': ['scale', 0.01, 0.1, 1]
}
# 4. Set up GridSearchCV.
grid_sv = GridSearchCV(
    estimator=pipe, param_grid=param_grid_sv, scoring=scoring, refit='recall_macro', cv=5, n_jobs=-1, verbose=2)

# 5. Fit the grid search to the training data.
grid_sv.fit(X_train, y_train)

# 6. Extract performances of different hyperparameter combinations.
cv_results_sv = pd.DataFrame(grid_sv.cv_results_).sort_values(
    by=['rank_test_recall_macro'])

cv_results_sv = cv_results_sv.set_index(
    'params').rename_axis('Hyperparameters')

cv_results_sv = cv_results_sv[cols_to_show]

print("--- Performances of different Hyperparameter Combinations ---")
print(cv_results_sv)

print("\nBest Hyperparameters:", grid_sv.best_params_)
print(
    "Best Cross-Validation Score (recall_macro): {:.2f}".format(grid_sv.best_score_))

---

# 3. **Evaluate models**

Here, you need to:

1.	test the model (the best one you obtained from the above stage) on the appropriate set.


In [ ]:
# Test the best model (RandomForestClassifier) with the best hyperparameters found in the grid search on the test set and report the performance metrics (accuracy, precision, recall, F1-score).

# Get the best model from grid search.
best_rf = grid_rf.best_estimator_

# Predict on the test set.
y_pred_rf = best_rf.predict(X_test)

# Evaluate the performance of the model on the test set using classification report and confusion matrix.
print("Macro-averaged Recall:", recall_score(y_test, y_pred_rf, average='macro'))
print("Macro-averaged Precision:", precision_score(y_test, y_pred_rf, average='macro'))
print("Macro-averaged F1-score:", f1_score(y_test, y_pred_rf, average='macro'))

# Print classification report.
print("Random Forest Classifier Performance on Test Set:")
print('(0: CI, 1: Depression, 2: Both)')
print(classification_report(y_test, y_pred_rf))

# Visualize the confusion matrix.
cm = confusion_matrix(y_test, y_pred_rf, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(4, 3))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['CI', 'Depression', 'Both'])
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')

plt.title('Confusion Matrix of the Final Random Forest Model', fontsize=10, fontweight='bold')
plt.xlabel('Predicted Diagnosis')
plt.ylabel('Actual Diagnosis')
plt.show()